In [ ]:
import os, sys
from IPython.display import Markdown, display
repo_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
code_path = os.path.abspath(os.path.join(repo_path, 'code/src'))
os.chdir(repo_path)
if code_path not in sys.path:
    sys.path.append(code_path)
display(Markdown(f'**Repository root:** {repo_path}'))

# ODD Protocol for the Pneumococcal ABM


## Table of Contents

1. [Overview](#overview)
2. [Design Concepts](#design-concepts)
3. [Details](#details)
4. [Code-Verified Single Timestep Trace](#single-timestep-trace)


# 1. Overview <a name="overview"></a>

## 1.1 Purpose

The model is an individual-level, age-structured, multi-serotype transmission model with vaccination and disease outcomes.
The class hierarchy is `Disease` → `VaryingTransmissionDisease` → `DiseaseModel`,
combining demography with serotype-group-specific transmission:
[model/disease/disease.py](../model/disease/disease.py) (line 21),
[model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (line 25),
[model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 115).

## 1.2 Entities, state variables, and scales

### Entities
- Individual agents are rows of `P.I` (a Polars DataFrame) in `DisPopulation`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 15).
- The disease process is managed by `VaryingTransmissionDisease` and wrapped by `DiseaseModel` which adds observers:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (line 25),
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 41).

### Agent-level state variables (per individual)
- Demography: `age`, `age_days`, `days_at_death`, `age_group`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 31).
- Stochastic individual heterogeneity: `quantile`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 31).
- Vaccination struct: `no_of_doses`, `on_time`, `vaccine_type`, `final_vaccine_time`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 31).
- Infection history / current carriage: `no_of_strains`, `strain_list`, `endTimes`, `no_past_infections`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 31).

### Time scale
- The model runs in discrete ticks; one tick is `364 // t_per_year` days:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 226).
- Baseline parameters define weekly time steps with `t_per_year = 52`:
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).

### Space / contact structure
- Interaction is age-mixing through a contact matrix loaded at runtime from
  `data/population/all_contact_matrix_Australia_prem_2017.csv` and wrapped in `KnownContactMatrix`:
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 74),
  [model/disease/contact_matrix.py](../model/disease/contact_matrix.py).

## 1.3 Process overview and scheduling

At each simulation tick in `DisSimulation._main_loop`, the order is:

1. Convert tick to day:
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 115).
2. Update demography (ageing / survival / births / migration):
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 137).
3. Call `Disease.update()` (which delegates FOI to `VaryingTransmissionDisease.calc_foi`):
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 115).

Inside `Disease.update()`, the disease-related sequence is:

1. Vaccination rollout checks (`check_vaccines`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 515, 534).
2. Community FOI + exposure sampling (overridden `calc_foi` from `VaryingTransmissionDisease`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 521, 927).
3. External introductions:
   [model/disease/disease.py](../model/disease/disease.py) (lines 525, 1354).
4. Individual state update for new infections:
   [model/disease/disease.py](../model/disease/disease.py) (lines 528, 1468).
5. Observer recording:
   [model/disease/disease.py](../model/disease/disease.py) (line 1478).


# 2. Design Concepts <a name="design-concepts"></a>

## 2.1 Basic principles

- Transmission is age-structured and **serotype-group-specific**.
  Serotypes are classified into four vaccine groups — pcv7, pcv13, ppv23 (PPV23 serotypes not in PCV7/13),
  and nonppv23 — each with its own transmission multiplier from `base_params.py`:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 52, 54, 86).
- For each group, FOI is built from age-group infection fractions and the contact matrix,
  then transformed as $1-e^{-FOI}$:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 376, 411).
- Co-infection is constrained by `max_no_coinfections`; susceptibility is reduced for already-infected individuals:
  [model/disease/disease.py](../model/disease/disease.py) (line 927).
- Antibody waning and protection are computed via the `antibody_levels` module:
  [model/disease/disease.py](../model/disease/disease.py) (line 20),
  [model/disease/antibody_levels.py](../model/disease/antibody_levels.py).

## 2.2 Emergence

- Serotype prevalence emerges from group-specific FOI-driven transmission plus external seeding
  and co-infection constraints (`no_of_strains` limit):
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 448, 472).
- IPD/CAP incidence by age and vaccine strata is an emergent output recorded by observers:
  [model/disease/disease.py](../model/disease/disease.py) (line 1141),
  [model/observers/obs_disease_by_age.py](../model/observers/obs_disease_by_age.py).

## 2.3 Adaptation and objectives

- Agents do not explicitly optimize behavior. Their state changes are rule-driven
  (vaccination eligibility / schedule, infection / recovery, ageing, death, migration):
  [model/disease/disease.py](../model/disease/disease.py) (lines 534, 927, 1468),
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 212).

## 2.4 Learning and prediction

- No explicit learning or memory update policy is implemented beyond cumulative exposure tracking
  (`no_past_infections`) and vaccine history fields:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 31).
- Vaccine impact is modeled through antibody-mediated probabilistic protection functions
  rather than adaptive policy learning:
  [model/disease/disease.py](../model/disease/disease.py) (lines 101, 127, 210).

## 2.5 Sensing and interaction

- Individuals do not directly sense neighbors; interaction is mediated through age-contact mixing
  and **group-specific** prevalence (pcv7/pcv13/ppv23/nonppv23) in the FOI calculation:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 411, 448).
- Vaccination interaction with infection occurs via vaccine history and serotype-specific
  antibody tables joined at exposure / outcome steps:
  [model/disease/disease.py](../model/disease/disease.py) (lines 927, 1141).

## 2.6 Stochasticity

- Randomness is included in infection seeding, exposure, strain assignment, vaccine timing,
  infection durations, and disease outcomes:
  [model/disease/disease.py](../model/disease/disease.py) (lines 286, 927, 1354, 1404, 1468).

## 2.7 Observation

`DiseaseModel` attaches observers for population, prevalence, vaccination rollout scenarios,
disease by age, vaccines delivered, and prevalence by age:
[run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (lines 50, 62).

Examples of recorded outputs:

- Population size and age distribution:
  [model/observers/obs_pop.py](../model/observers/obs_pop.py).
- Overall prevalence and serotype fractions:
  [model/observers/obs_prevalence.py](../model/observers/obs_prevalence.py).
- Vaccination rollout scenarios:
  [model/observers/obs_vacc_rollout_scenarios.py](../model/observers/obs_vacc_rollout_scenarios.py).
- Vaccines delivered by type:
  [model/observers/obs_vacc_delivered.py](../model/observers/obs_vacc_delivered.py).
- Age-specific infections and infected counts:
  [model/observers/obs_prevalence_by_age.py](../model/observers/obs_prevalence_by_age.py).
- Disease events by age group:
  [model/observers/obs_disease_by_age.py](../model/observers/obs_disease_by_age.py).


# 3. Details <a name="details"></a>

## 3.1 Initialization

- `go_single()` in `varying_transmission_run.py` creates output path / file naming based on the year span
  and either loads an existing disease file or creates a new one:
  [model/disease/varying_transmission_run.py](../model/disease/varying_transmission_run.py) (line 18).
- `DisSimulation.setup()` and `create_population()` build `DisPopulation`, set RNG / death rates,
  and generate the age-structured population:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 72, 89).
- When `read_population = True`, the initial population is restored from CSV triplets in
  `data/disease_pop_data/{pop_group}.csv`, `{pop_group}_endTimes.csv`, and `{pop_group}_strain_list.csv`:
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 31, 35).
- Initial infection seeding uses an age-dependent probability (doubled for children <= 2 years)
  and samples serotypes from the initialization strain distribution:
  [model/disease/disease.py](../model/disease/disease.py) (lines 286, 318).

## 3.2 Input data

- Baseline model parameters are in `run_scenarios/base_params.py`
  (demography, transmission, vaccination, seeds, time scale, run horizon):
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).
- Contact matrix is loaded at runtime from
  `data/population/all_contact_matrix_Australia_prem_2017.csv` as a NumPy array:
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 74).
- Strain, vaccine, age-specific protection, disease multipliers, and infection duration tables
  are loaded during `Disease._load_disease_data()`:
  [model/disease/disease.py](../model/disease/disease.py) (line 87).
- Per-serotype-group transmission multipliers are read from `p['transmission_coefficient_multipliers']`
  in `VaryingTransmissionDisease._load_disease_data()`:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 38, 52).

## 3.3 Submodels

### 3.3.1 Demography submodel

- Ages advance by `period = 364 // t_per_year` days per tick;
  individuals exceeding `days_at_death` are filtered out:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 226).
- Birth and migration flows are computed each tick from per-tick rates (`birth_rates[t]`, `mig_rates[t]`),
  with fractional residue accumulation for births:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 212).
- New births and migrants are added via `introduce_births_and_migrations()`;
  births are age 0 and migrants are sampled from the migration age distribution:
  [model/disease/disease_utils.py](../model/disease/disease_utils.py).

### 3.3.2 Vaccination submodel

- Vaccine list rows are parsed (including JSON-like list fields), expanded to full year coverage
  vectors, and aligned to the timestep grid:
  [model/disease/disease_utils.py](../model/disease/disease_utils.py).
- At each tick each active vaccine schedule computes the current rollout year and applies
  on-time / late coverage with vaccine-specific targeting (`check_vaccines`):
  [model/disease/disease.py](../model/disease/disease.py) (line 534).
- Late doses are represented through negative `on_time` values encoding future vaccination day:
  [model/disease/disease.py](../model/disease/disease.py) (line 534).

### 3.3.3 Transmission and acquisition submodel

- Serotypes are reclassified into four groups during initialization:
  **pcv7** (7 original PCV7 serotypes), **pcv13** (6 PCV13 additions),
  **ppv23** (PPV23 serotypes not in PCV7 or PCV13), **nonppv23** (all remaining serotypes):
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 54, 69, 86).
- Each group has a transmission multiplier from `p['transmission_coefficient_multipliers']`;
  a per-serotype multiplier lookup is built for use at runtime:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 93, 100).
- FOI per group is `transmission_coef x multiplier x sum(inf_fraction x contact_matrix_row)` per age group,
  computed by the overridden `calc_age_group_fois()`:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 376, 395).
- Infection fractions per group are computed by mapping each serotype to its group label
  and aggregating carrier counts (`calc_foi` override):
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 411, 448).
- Probability of exposure per age group is transformed as $1-e^{-FOI}$,
  then used in age-group binomial draws (`check_exposure`):
  [model/disease/disease.py](../model/disease/disease.py) (line 927).
- Eligibility excludes individuals already at `max_no_coinfections`;
  susceptibility is reduced via `1 - reduction_in_susceptibility x no_of_strains` clipped at 0:
  [model/disease/disease.py](../model/disease/disease.py) (line 927).
- If vaccinated, acquisition probability is gated by antibody waning and a logistic protection expression:
  [model/disease/disease.py](../model/disease/disease.py) (lines 101, 127).

### 3.3.4 Infection duration and recovery submodel

- Infection duration is sampled from an age-specific exponential distribution
  (`generate_duration_of_infection`):
  [model/disease/disease.py](../model/disease/disease.py) (line 1404).
- Carriage end times are appended on infection; expired infections are removed by
  filtering `endTimes > day` and updating `no_of_strains`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1468, 1474).

### 3.3.5 Disease outcome submodel

- New infections are passed to `check_disease()` where vaccine-antibody and age-protection data
  are joined with age-serotype disease multipliers:
  [model/disease/disease.py](../model/disease/disease.py) (line 1141).
- Disease probability is computed from log-antibodies;
  final disease draw uses `prob_of_disease x dis_multiplier`.
- Outcomes are split into `ipd` vs `cap` using age-specific `ipd_fraction_by_age_group`.

### 3.3.6 External exposure submodel

- At intervals defined by `external_exposure_check_per_year`, random external strains are
  sampled and assigned to susceptible unvaccinated individuals with probability `external_exposure_prob`:
  [model/disease/disease.py](../model/disease/disease.py) (line 1354).
- External infections go through the same duration assignment and state update as internal transmission:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1404, 1468).

### 3.3.7 Population save / load submodel

- When `save_population = True`, the population state is serialised to CSV triplets:
  `{pop_saving_address}.csv`, `{pop_saving_address}_endTimes.csv`,
  and `{pop_saving_address}_strain_list.csv`:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 189).
- When `read_population = True`, the saved state is restored from
  `data/disease_pop_data/{pop_group}.csv` and companion files,
  bypassing synthetic population generation:
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 31, 35).
- The `pop_group` and `pop_saving_address` parameters in `base_params.py` control which checkpoint files
  are read / written:
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).


# 4. Code-Verified Single Timestep Trace <a name="single-timestep-trace"></a>

Given tick `t`, with `day = t * 364 // t_per_year`, one full iteration does:

1. **Demography update** (if enabled): agents age by `period = 364 // t_per_year` days,
   deaths removed, births and migrants added.
   Sources: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py)
   (lines 115, 137, 212, 226).

2. **Vaccination update**: current schedules evaluated, on-time and late doses assigned,
   agent vaccine struct updated.
   Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 515, 534).

3. **FOI and exposure sampling**: age-group exposures drawn per vaccine group
   (pcv7 / pcv13 / ppv23 / nonppv23) via the overridden `calc_foi()` and `calc_age_group_fois()`,
   exposed strain sampled from group infection distribution,
   vaccine-mediated acquisition filtering applied.
   Sources: [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py)
   (lines 376, 411, 448),
   [model/disease/disease.py](../model/disease/disease.py) (lines 521, 927).

4. **External introductions** (if tick is due): imported strains sampled and applied to eligible hosts.
   Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 525, 1354).

5. **Individual state updates**: newly infected individuals' `strain_list` and `endTimes` updated;
   infection duration drawn from age-specific exponential.
   Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 528, 1404, 1468).

6. **Disease outcomes** for newly infected: probability-based CAP / IPD outcomes sampled,
   recorded into `P.disease_pop` and observer channels.
   Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 528, 1141).

7. **Recovery**: infections with `endTimes <= day` removed from each agent.
   Source: [model/disease/disease.py](../model/disease/disease.py) (lines 1468, 1474).

8. **Observer writes**: prevalence, disease, vaccine, and population summaries persisted.
   Source: [model/disease/disease.py](../model/disease/disease.py) (line 1478),
   [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (lines 50, 62).

This sequence is repeated for all ticks in the configured simulation horizon
(`years = [start, end]`, `t_per_year = 52`):
[run_scenarios/base_params.py](../run_scenarios/base_params.py),
[model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 115).
